# LNL-MoEx-Ti: accuracy-first training

This variant keeps only the low-risk boosting analogies: bounded updates, exact-count early stopping, and a short MoEx-free finishing phase. Persistent hard-example weights are intentionally excluded because augmentation makes per-sample loss noisy and adds checkpoint complexity without evidence of a clean Top-1 gain.

In [ ]:
from pathlib import Path
import os
import subprocess

# Pipeline 1/7 - Locate the local repository or clone it when needed.
REPOSITORY_URL = "https://github.com/Omid-Nejati/Locality-iN-Locality.git"
repository_root = next(
    (
        path
        for path in (
            Path.cwd(),
            Path.cwd() / "Locality-iN-Locality-main",
            Path.cwd() / "Locality-iN-Locality",
        )
        if (path / "LNL_MoEx.py").exists()
    ),
    None,
)

if repository_root is None:
    repository_root = Path.cwd() / "Locality-iN-Locality"
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(repository_root)],
        check=True,
    )

os.chdir(repository_root)
print("Working directory:", Path.cwd())

In [ ]:
%pip install -q timm==0.9.16 einops scikit-learn

In [ ]:
import csv
import math
import random
import shutil
import zipfile
from dataclasses import dataclass
from urllib.request import urlretrieve

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from timm.utils import ModelEmaV2
from torch.utils.data import DataLoader, Subset

SEED = 42

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = device.type == "cuda"

print(
    f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | "
    f"timm {timm.__version__} | device {device}"
)

## Data

In [ ]:
# Pipeline 2/7 - Download, extract, and arrange the official GTSRB split.
DATA_DIR = Path("data")
ARCHIVE_BASE_URL = (
    "https://sid.erda.dk/public/archives/"
    "daaeac0d7ce1152aea9b61d9f1e19370"
)
ARCHIVES = (
    "GTSRB_Final_Training_Images.zip",
    "GTSRB_Final_Test_Images.zip",
    "GTSRB_Final_Test_GT.zip",
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
required_paths = (
    DATA_DIR / "GTSRB" / "Final_Training" / "Images",
    DATA_DIR / "GTSRB" / "Final_Test" / "Images",
    DATA_DIR / "GT-final_test.csv",
)
if not all(path.exists() for path in required_paths):
    for filename in ARCHIVES:
        destination = DATA_DIR / filename
        if not destination.exists():
            print("Downloading", filename)
            urlretrieve(f"{ARCHIVE_BASE_URL}/{filename}", destination)
    for filename in ARCHIVES:
        with zipfile.ZipFile(DATA_DIR / filename) as archive:
            archive.extractall(DATA_DIR)
    print("GTSRB archives extracted.")

test_source = DATA_DIR / "GTSRB" / "Final_Test" / "Images"
test_root = DATA_DIR / "GTSRB" / "test"
test_root.mkdir(parents=True, exist_ok=True)

with (DATA_DIR / "GT-final_test.csv").open(newline="") as csv_file:
    test_rows = list(csv.DictReader(csv_file, delimiter=";"))

copied = 0
for row in test_rows:
    destination = test_root / f"{int(row['ClassId']):04d}" / row["Filename"]
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(test_source / row["Filename"], destination)
        copied += 1

print(f"Official test layout ready: {len(test_rows)} images ({copied} copied).")

In [ ]:
# Pipeline 3/7 - Keep sign geometry and randomize training images only.
class ResizeWithPadding:
    def __init__(self, size=224, fill=128):
        self.size = size
        self.fill = fill

    def __call__(self, image: Image.Image):
        width, height = image.size
        scale = self.size / max(width, height)
        resized_width = max(1, round(width * scale))
        resized_height = max(1, round(height * scale))

        image = TF.resize(
            image,
            [resized_height, resized_width],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )
        pad_width = self.size - resized_width
        pad_height = self.size - resized_height
        padding = [
            pad_width // 2,
            pad_height // 2,
            pad_width - pad_width // 2,
            pad_height - pad_height // 2,
        ]
        return TF.pad(image, padding, fill=self.fill)

NORMALIZE = transforms.Normalize((0.5,) * 3, (0.5,) * 3)
train_transform = transforms.Compose([
    ResizeWithPadding(),
    transforms.RandAugment(
        num_ops=2,
        magnitude=7,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=128,
    ),
    transforms.ToTensor(),
    NORMALIZE,
])
eval_transform = transforms.Compose([
    ResizeWithPadding(),
    transforms.ToTensor(),
    NORMALIZE,
])

In [ ]:
# Pipeline 4/7 - Build a track-safe validation split and data loaders.
TRAIN_ROOT = DATA_DIR / "GTSRB" / "Final_Training" / "Images"
TEST_ROOT = DATA_DIR / "GTSRB" / "test"
TARGET_VALIDATION_SIZE = 4000
MICRO_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 2

index_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT)
indices = np.arange(len(index_dataset))
targets = np.asarray(index_dataset.targets)


def track_id(sample_path):
    path = Path(sample_path)
    parts = path.stem.split("_")
    if len(parts) != 2 or not all(part.isdigit() for part in parts):
        raise ValueError(
            f"Unexpected GTSRB filename {path.name!r}; "
            "expected <TrackID>_<FrameID>.ppm."
        )
    return f"{path.parent.name}:{parts[0]}"


groups = np.asarray([track_id(path) for path, _ in index_dataset.samples])
min_tracks_per_class = min(
    np.unique(groups[targets == class_id]).size
    for class_id in range(len(index_dataset.classes))
)
n_splits = min(
    max(2, round(len(index_dataset) / TARGET_VALIDATION_SIZE)),
    min_tracks_per_class,
)
if n_splits < 2:
    raise ValueError("At least two distinct tracks per class are required.")

splitter = StratifiedGroupKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED,
)
train_indices, validation_indices = min(
    splitter.split(indices, targets, groups),
    key=lambda split: abs(len(split[1]) - TARGET_VALIDATION_SIZE),
)

train_groups = set(groups[train_indices])
validation_groups = set(groups[validation_indices])
assert train_groups.isdisjoint(validation_groups)
assert len(train_indices) + len(validation_indices) == len(index_dataset)

trainset = Subset(
    torchvision.datasets.ImageFolder(TRAIN_ROOT, transform=train_transform),
    train_indices,
)
validationset = Subset(
    torchvision.datasets.ImageFolder(TRAIN_ROOT, transform=eval_transform),
    validation_indices,
)
testset = torchvision.datasets.ImageFolder(TEST_ROOT, transform=eval_transform)


def make_loader(dataset, batch_size, *, shuffle=False, drop_last=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )


train_loader = make_loader(
    trainset,
    MICRO_BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
validation_loader = make_loader(validationset, EVAL_BATCH_SIZE)
test_loader = make_loader(testset, EVAL_BATCH_SIZE)
NUM_CLASSES = len(index_dataset.classes)

for label, value in {
    "Training images": len(trainset),
    "Validation images": len(validationset),
    "Training tracks": len(train_groups),
    "Validation tracks": len(validation_groups),
    "Test images": len(testset),
    "Classes": NUM_CLASSES,
}.items():
    print(f"{label}: {value}")

## Train LNL-MoEx-Ti

In [ ]:
from LNL_MoEx import LNL_MoEx_Ti

# XGBoost transfer 1/2 - Shrinkage: small LR, warmup, and cosine decay.
NUM_EPOCHS = 60
MIN_EPOCHS = 35
EARLY_STOP_PATIENCE = 8
TARGET_VALIDATION_TOP1 = 99.5
TARGET_STABLE_EPOCHS = 2

WARMUP_EPOCHS = 5
ACCUMULATION_STEPS = 4
LEARNING_RATE = 3e-4
MIN_LR_FACTOR = 0.01
WEIGHT_DECAY = 0.05

# DL support - MoEx/EMA improve generalization; they are not XGBoost ideas.
MOEX_LAMBDA = 0.90
MOEX_PROBABILITY = 0.35
MOEX_STOP_EPOCH = 30
EMA_DECAY = 0.9998

default_checkpoint_dir = (
    Path("/content/drive/MyDrive/LNL_MoEx_checkpoints")
    if Path("/content/drive/MyDrive").exists()
    else Path("checkpoints")
)
CHECKPOINT_DIR = Path(
    os.environ.get("CHECKPOINT_DIR", str(default_checkpoint_dir))
).expanduser()
CHECKPOINT_PATH = CHECKPOINT_DIR / (
    f"LNL_MoEx_RandAug_Ti_GTSRB_fresh_v5_accuracy_first_seed{SEED}_top1.pth"
)
RESUME_TRAINING = os.environ.get(
    "RESUME_TRAINING", "0"
).strip().lower() in {"1", "true", "yes", "on"}
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def build_model():
    return LNL_MoEx_Ti(
        pretrained=False,
        num_classes=NUM_CLASSES,
        drop_path_rate=0.10,
    ).to(device)


model = build_model()
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

decay_parameters = []
no_decay_parameters = []
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    no_decay = (
        parameter.ndim == 1
        or name.endswith(".bias")
        or name in {"cls_token", "patch_pos", "pixel_pos"}
    )
    (no_decay_parameters if no_decay else decay_parameters).append(parameter)

optimizer = optim.AdamW(
    [
        {"params": decay_parameters, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
)


# Shrink each update progressively after warmup.
def learning_rate_factor(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(
        1, NUM_EPOCHS - WARMUP_EPOCHS
    )
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=learning_rate_factor,
)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
ema_model = ModelEmaV2(model, decay=EMA_DECAY)

print("Training mode:", "resume" if RESUME_TRAINING else "fresh start")
print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
# XGBoost transfer 2/2 - Persist best score and early-stop counters.
@dataclass
class TrainState:
    epoch: int = 0
    best_correct: int = -1
    best_top1: float = -1.0
    best_weights: dict = None
    stale_epochs: int = 0
    target_streak: int = 0


# Compare exact correct counts so one-image gains are never rounded away.
@torch.inference_mode()
def evaluate_top1(module, loader):
    module.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            predictions = module(images).argmax(dim=1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)

    if not total:
        raise ValueError("Cannot evaluate an empty loader.")
    return correct, total, 100.0 * correct / total

In [ ]:
def cpu_state_dict(module):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in module.state_dict().items()
    }


def load_torch_file(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


# Pipeline 5/7 - Save every state required for an exact training resume.
def save_checkpoint(state):
    best_weights = state.best_weights
    if best_weights is None:
        best_weights = cpu_state_dict(ema_model.module)

    payload = {
        "format_version": 5,
        "epoch": state.epoch,
        "best_validation_correct": state.best_correct,
        "best_validation_top1": state.best_top1,
        "no_improvement_epochs": state.stale_epochs,
        "target_streak": state.target_streak,
        "model_state_dict": model.state_dict(),
        "ema_model_state_dict": ema_model.module.state_dict(),
        "best_ema_model_state_dict": best_weights,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "ema_decay": EMA_DECAY,
        "rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
    }
    if torch.cuda.is_available():
        payload["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

    temporary_path = CHECKPOINT_PATH.with_suffix(".pth.tmp")
    torch.save(payload, temporary_path)
    os.replace(temporary_path, CHECKPOINT_PATH)


def load_checkpoint():
    if not RESUME_TRAINING or not CHECKPOINT_PATH.exists():
        return TrainState()

    checkpoint = load_torch_file(CHECKPOINT_PATH)
    if checkpoint.get("format_version") != 5:
        raise ValueError(
            "RESUME_TRAINING requires a format-version 5 checkpoint."
        )

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    ema_model.module.load_state_dict(
        checkpoint["ema_model_state_dict"], strict=True
    )
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    for optimizer_state in optimizer.state.values():
        for key, value in optimizer_state.items():
            if torch.is_tensor(value):
                optimizer_state[key] = value.to(device)

    torch.set_rng_state(checkpoint["rng_state"].cpu())
    np.random.set_state(checkpoint["numpy_rng_state"])
    random.setstate(checkpoint["python_rng_state"])
    if torch.cuda.is_available() and "cuda_rng_state_all" in checkpoint:
        torch.cuda.set_rng_state_all([
            state.cpu()
            for state in checkpoint["cuda_rng_state_all"]
        ])

    state = TrainState(
        epoch=int(checkpoint["epoch"]),
        best_correct=int(checkpoint["best_validation_correct"]),
        best_top1=float(checkpoint["best_validation_top1"]),
        best_weights=checkpoint["best_ema_model_state_dict"],
        stale_epochs=int(checkpoint["no_improvement_epochs"]),
        target_streak=int(checkpoint["target_streak"]),
    )

    print(
        f"Resumed at epoch {state.epoch}/{NUM_EPOCHS}; "
        f"best validation: {state.best_correct}/{len(validationset)} "
        f"({state.best_top1:.3f}%)"
    )
    return state

In [ ]:
# Deliberate omission - No persistent hard-example/sample weighting.
# RandAugment and MoEx make per-image loss too noisy for that extra state.
def compute_loss(images, labels, *, use_moex):
    with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
        if not use_moex:
            return criterion(model(images), labels)

        swap_index = torch.randperm(images.size(0), device=images.device)
        logits = model(
            images,
            swap_index=swap_index,
            moex_norm="pono",
            moex_epsilon=1e-5,
            moex_layer="stem",
            moex_positive_only=False,
        )
        return (
            MOEX_LAMBDA * criterion(logits, labels)
            + (1.0 - MOEX_LAMBDA)
            * criterion(logits, labels[swap_index])
        )


def train_one_epoch(epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    total_steps = len(train_loader)
    # DL support - Finish with MoEx off to stabilize the clean boundary.
    allow_moex = epoch < MOEX_STOP_EPOCH
    update_ema = epoch >= WARMUP_EPOCHS

    for step, (images, labels) in enumerate(train_loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        use_moex = (
            allow_moex
            and torch.rand(1).item() < MOEX_PROBABILITY
        )
        raw_loss = compute_loss(images, labels, use_moex=use_moex)
        total_loss += raw_loss.item()
        group_start = ((step - 1) // ACCUMULATION_STEPS) * ACCUMULATION_STEPS
        group_size = min(ACCUMULATION_STEPS, total_steps - group_start)
        scaler.scale(raw_loss / group_size).backward()

        if step % ACCUMULATION_STEPS and step != total_steps:
            continue

        scaler.unscale_(optimizer)
        # Shrinkage guard - Bound unusually large optimizer updates.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if update_ema and scaler.get_scale() >= previous_scale:
            ema_model.update(model)
        optimizer.zero_grad(set_to_none=True)

    return total_loss / total_steps, allow_moex


# Early stopping starts only after the minimum training budget.
def stopping_reason(state):
    if state.epoch < MIN_EPOCHS:
        return None
    if state.target_streak >= TARGET_STABLE_EPOCHS:
        return f"target > {TARGET_VALIDATION_TOP1:.2f}% was stable"
    if state.stale_epochs >= EARLY_STOP_PATIENCE:
        return (
            f"no validation improvement for "
            f"{EARLY_STOP_PATIENCE} epochs"
        )
    return None

In [ ]:
# Pipeline 6/7 - Train, validate, checkpoint, and stop on stable progress.
state = load_checkpoint()

for epoch in range(state.epoch, NUM_EPOCHS):
    training_loss, moex_active = train_one_epoch(epoch)

    if epoch < WARMUP_EPOCHS:
        ema_model.module.load_state_dict(
            model.state_dict(),
            strict=True,
        )

    correct, total, top1 = evaluate_top1(
        ema_model.module,
        validation_loader,
    )
    # Early stopping uses exact correct count, not rounded Top-1.
    improved = correct > state.best_correct
    if improved:
        state.best_correct = correct
        state.best_top1 = top1
        state.best_weights = cpu_state_dict(ema_model.module)
        state.stale_epochs = 0
    else:
        state.stale_epochs += 1

    state.target_streak = (
        state.target_streak + 1
        if top1 > TARGET_VALIDATION_TOP1
        else 0
    )
    state.epoch = epoch + 1
    # Apply the shrinkage schedule once after each completed epoch.
    scheduler.step()
    save_checkpoint(state)

    print(
        f"Epoch {state.epoch:03d}/{NUM_EPOCHS} | "
        f"loss {training_loss:.4f} | "
        f"MoEx {'on' if moex_active else 'off'} | "
        f"val {correct}/{total} ({top1:.3f}%) | "
        f"best {state.best_correct}/{total} "
        f"({state.best_top1:.3f}%) | "
        f"plateau {state.stale_epochs}/{EARLY_STOP_PATIENCE}"
    )

    reason = stopping_reason(state)
    if reason:
        print("Early stopping:", reason)
        break

print(
    f"Best validation: {state.best_correct}/{len(validationset)} "
    f"({state.best_top1:.3f}%)"
)

## Official test

In [ ]:
# Pipeline 7/7 - Load the best EMA weights and test exactly once.
checkpoint = load_torch_file(CHECKPOINT_PATH)
weights = checkpoint["best_ema_model_state_dict"]

verification_model = build_model()
verification_model.load_state_dict(weights, strict=True)
test_correct, test_total, test_top1 = evaluate_top1(
    verification_model,
    test_loader,
)

print(
    f"Top-1 accuracy: {test_correct}/{test_total} "
    f"({test_top1:.3f}%)"
)
print(
    f"Target > {TARGET_VALIDATION_TOP1:.1f}%: "
    f"{'achieved' if test_top1 > TARGET_VALIDATION_TOP1 else 'not achieved'}."
)

try:
    from google.colab import files
except ImportError:
    print("Checkpoint:", CHECKPOINT_PATH.resolve())
else:
    files.download(str(CHECKPOINT_PATH))